# Support Vector Machines (SVMs) for Grain Disease Recognition
Previously we used Logistic Regression to draw simple lines between healthy and infected crops. 

Now, we're upgrading to the second classical supervised machine learning model via deploying Support Vector Machines (SVMs) to find complex boundaries in our data.

## What are SVMs?
It finds the optimal hyperplane that separates classes with the maximum possible margin. The data points sitting closest to this boundary are called support vectors, and they are the only points that actually determine where the boundary is drawn.

For cases where classes are not linearly separable, the kernel trick implicitly maps features into a higher-dimensional space where a separating hyperplane can be found. We use the Radial Basis Function (RBF) kernel, which measures similarity between points.

**If you didn't understand:** instead of drawing a straight line between disease classes, the SVM draws complex curved boundaries that can wrap around clusters of similar grains in feature space.

## Why use SVMs?
Building off of the limitation of Logistic Regression only drawing straight lines between classes, SVM removes that constraint via its kernel trick!

# Setup

Making the necessary imports.

In [2]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import os
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    classification_report,
)
from tqdm import tqdm
import joblib
import pandas as pd

from logistic_regression_lib import collect_embeddings

# Data Preprocessing
Before we can train our model we must define the world it will operate in. 

We will define these lists so our system knows exactly what to look for.

In [3]:
GRAIN_TYPES = ["maize", "rice"]
MAIZE_CATEGORIES = ["0_NOR", "1_F&S", "2_SD", "3_MY", "4_AP", "5_BN", "6_HD"]
RICE_CATEGORIES = ["0_NOR", "1_F&S", "2_SD", "3_MY", "4_AP", "5_BN", "6_UN"]

By using our feature extraction script we analyze the HSV color space to understand the exact hues of the grain surface. Same as in Logistic Regression.

We also compute Histogram of Oriented Gradients (HOG) features to capture the physical texture and structural damage of the crops. This transforms every photograph into a dense 512 number feature vector. 

**Intuition:** If the cracks of a maize image run in specific directions, HOG captures that and not the colors.

In [4]:
# Collect training data
X_train_maize, y_train_maize = collect_embeddings("maize", "train")
X_train_rice, y_train_rice = collect_embeddings("rice", "train")
X_val_maize, y_val_maize = collect_embeddings("maize", "val")
X_val_rice, y_val_rice = collect_embeddings("rice", "val")

# Combine everything into our main training set
X_train = np.concatenate((X_train_maize, X_train_rice, X_val_maize, X_val_rice))
y_train = np.concatenate((y_train_maize, y_train_rice, y_val_maize, y_val_rice))

print(f"\nTraining data shape: {X_train.shape}")
print(f"Number of classes: {len(set(y_train))}")

Processing train images: 100%|██████████| 12240/12240 [02:46<00:00, 73.45it/s] 


Processing train images: 100%|██████████| 22155/22155 [01:32<00:00, 239.60it/s]


Processing val images: 100%|██████████| 2160/2160 [00:32<00:00, 66.33it/s] 


Processing val images: 100%|██████████| 3907/3907 [00:16<00:00, 235.99it/s]



Training data shape: (40462, 512)
Number of classes: 14


We cannot evaluate our model using the same images it studied during training. We must gather a completely separate set of photographs to serve as our final test.

In [5]:
X_test_maize, y_test_maize = collect_embeddings("maize", "test")
X_test_rice, y_test_rice = collect_embeddings("rice", "test")
X_test = np.concatenate((X_test_maize, X_test_rice))
y_test = np.concatenate((y_test_maize, y_test_rice))

print(f"Test data shape: {X_test.shape}")

Processing test images: 100%|██████████| 1600/1600 [00:17<00:00, 92.84it/s] 


Processing test images: 100%|██████████| 2900/2900 [00:08<00:00, 337.24it/s]

Test data shape: (4500, 512)


To make the math work seamlessly we need to convert our text labels into numerical codes. The Label Encoder handles this.

In [6]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Constructing the SVM Pipeline
Here is where the real magic happens. Support Vector Machines work by projecting our data into higher dimensions and finding the optimal hyperplane that separates the different diseases.

Before we can train our model we need to build the assembly line. This pipeline ensures every image feature vector is perfectly scaled before the algorithm even sees it. We also define the exact limits of our experiment by setting up a grid of possible settings for the machine to explore.

In [7]:
# Set up the scaling and model pipeline
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("classifier", SVC(random_state=42))]
)

# The parameter grid for our search
param_dist = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__kernel": ["linear", "rbf"],
    "classifier__gamma": ["scale", "auto", 0.1, 0.01],
    "classifier__class_weight": [None, "balanced"],
}

# Tuning Hyperparameters
We then take the pipeline and the parameters we just built and feed them into the randomized search. Instead of testing every single combination we use a randomized search to save computation time. The algorithm will blindly test 20 different configurations to learn which combination of mathematical boundaries best separates the healthy maize and rice from the diseased ones.

In [8]:
n_iter = 20

print("Starting Random Search for SVM...")

# Configure the search parameters
random_search = RandomizedSearchCV(
    pipeline,
    param_dist,
    n_iter=n_iter,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=2,
    random_state=42,
)

# Train the model and find the best settings
random_search.fit(X_train, y_train_encoded)

print("\nRANDOM SEARCH RESULTS")
print(f"\nBest hyperparameters: {random_search.best_params_}")
print(f"Best cross-validation F1-score: {random_search.best_score_:.4f}")

Starting Random Search for SVM...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

RANDOM SEARCH RESULTS

Best hyperparameters: {'classifier__kernel': 'rbf', 'classifier__gamma': 'scale', 'classifier__class_weight': None, 'classifier__C': 100}
Best cross-validation F1-score: 0.7862


The machine has finished its training phase! 

# Model Evaluation
Now, we pull the absolute best model from the search and pit it against our unseen test photographs. 

Then, we calculate the precision and recall to generate a comprehensive report card so we can see if our new algorithm outperformed our previous logistic regression efforts. 

In [ ]:
best_pipeline = random_search.best_estimator_

y_test_pred_best = best_pipeline.predict(X_test)

best_accuracy = best_pipeline.score(X_test, y_test_encoded)
best_f1 = f1_score(y_test_encoded, y_test_pred_best, average="macro")
best_recall = recall_score(y_test_encoded, y_test_pred_best, average="macro")
best_precision = precision_score(y_test_encoded, y_test_pred_best, average="macro")

print("\nTEST SET PERFORMANCE")
print(f"Test Accuracy:  {best_accuracy:.4f}")
print(f"Test F1-Score (macro):  {best_f1:.4f}")
print(f"Test Recall (macro):    {best_recall:.4f}")
print(f"Test Precision (macro): {best_precision:.4f}")

print("\nPER-CLASS METRICS")
best_report = classification_report(
    y_test_encoded, y_test_pred_best, target_names=label_encoder.classes_, digits=4
)
print(best_report)

# Securely saving the model
os.makedirs("models", exist_ok=True)
joblib.dump(best_pipeline, "models/svm_tuned.pkl")
print("\nBest tuned pipeline saved to models/svm_tuned.pkl")


TEST SET PERFORMANCE
Test Accuracy:  0.8811
Test F1-Score:  0.8817
Test Recall:    0.8811
Test Precision: 0.8874

PER-CLASS METRICS
              precision    recall  f1-score   support

 maize_0_NOR     0.9205    0.9500    0.9350      1000
 maize_1_F&S     0.9314    0.9500    0.9406       100
  maize_2_SD     0.9551    0.8500    0.8995       100
  maize_3_MY     0.6727    0.3700    0.4774       100
  maize_4_AP     0.6491    0.7400    0.6916       100
  maize_5_BN     0.9103    0.7100    0.7978       100
  maize_6_HD     0.9744    0.7600    0.8539       100
  rice_0_NOR     0.9729    0.9515    0.9621      2000
  rice_1_F&S     0.5614    0.6400    0.5981       150
   rice_2_SD     0.6821    0.6867    0.6844       150
   rice_3_MY     0.7945    0.7733    0.7838       150
   rice_4_AP     0.6464    0.7800    0.7069       150
   rice_5_BN     0.5879    0.7800    0.6705       150
   rice_6_UN     0.8446    0.8333    0.8389       150

    accuracy                         0.8811      4500
 

## Reloading the Model
Due to training taking long, we have the saved model file to load for any adjustments or checks.

In [8]:
import joblib

best_pipeline = joblib.load("models/svm_tuned.pkl")  # Load saved model

y_test_pred_best = best_pipeline.predict(X_test)

best_accuracy = best_pipeline.score(X_test, y_test_encoded)
best_f1 = f1_score(y_test_encoded, y_test_pred_best, average="macro")
best_recall = recall_score(y_test_encoded, y_test_pred_best, average="macro")
best_precision = precision_score(y_test_encoded, y_test_pred_best, average="macro")

print("\nTEST SET PERFORMANCE")
print(f"Test Accuracy:  {best_accuracy:.4f}")
print(f"Test F1-Score (macro):  {best_f1:.4f}")
print(f"Test Recall (macro):    {best_recall:.4f}")
print(f"Test Precision (macro): {best_precision:.4f}")

print("\nPER-CLASS METRICS")
best_report = classification_report(
    y_test_encoded, y_test_pred_best, target_names=label_encoder.classes_, digits=4
)
print(best_report)


TEST SET PERFORMANCE
Test Accuracy:  0.8811
Test F1-Score (macro):  0.7743
Test Recall (macro):    0.7696
Test Precision (macro): 0.7931

PER-CLASS METRICS
              precision    recall  f1-score   support

 maize_0_NOR     0.9205    0.9500    0.9350      1000
 maize_1_F&S     0.9314    0.9500    0.9406       100
  maize_2_SD     0.9551    0.8500    0.8995       100
  maize_3_MY     0.6727    0.3700    0.4774       100
  maize_4_AP     0.6491    0.7400    0.6916       100
  maize_5_BN     0.9103    0.7100    0.7978       100
  maize_6_HD     0.9744    0.7600    0.8539       100
  rice_0_NOR     0.9729    0.9515    0.9621      2000
  rice_1_F&S     0.5614    0.6400    0.5981       150
   rice_2_SD     0.6821    0.6867    0.6844       150
   rice_3_MY     0.7945    0.7733    0.7838       150
   rice_4_AP     0.6464    0.7800    0.7069       150
   rice_5_BN     0.5879    0.7800    0.6705       150
   rice_6_UN     0.8446    0.8333    0.8389       150

    accuracy                   

# Findings
The SVM with an RBF kernel successfully learned to distinguish between 14 grain disease classes, achieving a test accuracy of 0.8811 and a weighted F1 of 0.8817 across 4,500 unseen images.

## What went well?
The hyperparameter search selected C=100, kernel=rbf, gamma=scale as the best configuration. The RBF kernel's selection over linear confirms that the color feature space requires curved, non-linear boundaries to separate disease classes effectively. 

Several classes were identified reliably, maize_0_NOR (0.9350), rice_0_NOR (0.9621), maize_1_F&S (0.9406), and rice_6_UN (0.8389) all scored strongly.

## What went wrong?
maize_3_MY was the weakest class with an F1 of only 0.4774 and a recall of 0.37, meaning the model misses more than 60% of actual Moldy maize cases. rice_1_F&S (0.5981) and rice_5_BN (0.6705) also performed poorly on broken grains

The gap between weighted F1 (0.8817) and macro F1 (0.7743) reveals that strong overall numbers are being carried by the large normal classes. The model is significantly less reliable on the minority disease classes that matter most for practical grain quality inspection.

## What do these imply?
The consistent failure on certain disease classes points to a limitation in the feature representation rather than the classifier itself. The model only sees color, a 512-dimensional HSV histogram with no texture or spatial information. Classes that share similar color profiles but differ in surface damage patterns cannot be reliably separated by any classifier working on these features alone.

This suggests that richer feature representations, like texture descriptors or features learned directly from raw pixels, are needed to push performance further on the difficult minority classes.